In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_6.py ──
"""
Shared infrastructure for MLFP03 Exercise 6 — Interpretability and Fairness.

Contains: Singapore credit scoring data load, LightGBM model training,
TreeSHAP explainer setup, output directory, and common helper utilities.

Technique-specific code (permutation importance loops, LIME wrappers,
fairness audit reports) lives in the per-technique files under
`modules/mlfp03/solutions/ex_6/`.

Import pattern (solutions and local both):

"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl

import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input



# ════════════════════════════════════════════════════════════════════════
# PATHS / CONSTANTS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp03_ex6_interpretability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Singapore credit scoring: Monetary Authority of Singapore (MAS) requires
# explainability for credit decisions under the Model Risk Management
# guideline. This dataset simulates a retail-bank default prediction task
# used throughout MLFP02/MLFP03.
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"
RANDOM_SEED = 42

# Protected attribute candidates we audit for disparate impact.
PROTECTED_CANDIDATES: list[str] = ["age", "gender", "ethnicity", "marital_status"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOAD + MODEL TRAIN
# ════════════════════════════════════════════════════════════════════════

# Populated on first call so every technique file sees the same split.
_CACHE: dict[str, Any] = {}


def load_credit_scoring() -> dict[str, Any]:
    """Load the Singapore credit scoring dataset and run the M3 preprocessing
    pipeline. Returns a dict with X_train, y_train, X_test, y_test, feature_names.

    The return value is cached so repeated calls from different technique
    files re-use the same split (essential for interpretability comparisons).
    """
    if _CACHE:
        return _CACHE

    loader = MLFPDataLoader()
    credit: pl.DataFrame = loader.load(DATASET_MODULE, DATASET_FILE)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target=TARGET_COLUMN,
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_columns = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    feature_names: list[str] = col_info["feature_columns"]

    _CACHE.update(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=feature_names,
    )
    return _CACHE


def train_credit_model() -> dict[str, Any]:
    """Train the LightGBM credit default model. Cached per-process.

    Returns a dict with model, y_proba, y_pred, auc, and all data from
    `load_credit_scoring()`.
    """
    if "model" in _CACHE:
        return _CACHE

    data = load_credit_scoring()
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
        random_state=RANDOM_SEED,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)

    _CACHE.update(model=model, y_proba=y_proba, y_pred=y_pred, auc=auc)
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# SHAP EXPLAINER
# ════════════════════════════════════════════════════════════════════════


def build_shap_explainer() -> dict[str, Any]:
    """Construct the TreeSHAP explainer and compute SHAP values for X_test.

    Returns the full bundle: model, data, explainer, shap_vals, expected_value.
    """
    if "shap_vals" in _CACHE:
        return _CACHE

    bundle = train_credit_model()
    explainer = shap.TreeExplainer(bundle["model"])
    shap_values = explainer.shap_values(bundle["X_test"])

    # TreeSHAP for binary classifiers may return [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    expected_value = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, list)
        else explainer.expected_value
    )

    _CACHE.update(
        explainer=explainer,
        shap_vals=shap_vals,
        expected_value=expected_value,
    )
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# REUSABLE UTILITIES
# ════════════════════════════════════════════════════════════════════════


def rank_features_by_mean_abs_shap(
    shap_vals: np.ndarray, feature_names: list[str]
) -> list[tuple[str, float]]:
    """Return [(feature, mean_abs_shap), ...] sorted descending."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    return sorted(zip(feature_names, mean_abs), key=lambda x: x[1], reverse=True)


def feature_index(feature_names: list[str], name: str) -> int:
    """Lookup a feature column index by name, raising a clear error."""
    if name not in feature_names:
        raise KeyError(
            f"Feature '{name}' not found. Available: {feature_names[:10]}..."
        )
    return feature_names.index(name)


def synthetic_group_split(
    X: np.ndarray, feature_idx: int = 0
) -> tuple[np.ndarray, np.ndarray, float]:
    """Split X into two groups on a median cut of `feature_idx`.

    Returns (group_a_mask, group_b_mask, median_value).
    Used as a fallback when no protected attribute is present in features.
    """
    vals = X[:, feature_idx]
    median_val = float(np.median(vals))
    group_a = vals <= median_val
    group_b = ~group_a
    return group_a, group_b, median_val


def print_section(title: str, char: str = "=") -> None:
    """Print a standardised section banner."""
    line = char * 70
    print(f"\n{line}")
    print(f"  {title}")
    print(line)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 6.3: LIME Local Explanations + SHAP Waterfalls
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Use LIME to fit a local linear surrogate around a single prediction
#   - Compare LIME's local weights against SHAP for the same sample
#   - Explain the HIGHEST-risk and LOWEST-risk applications in the test set
#   - Build the "right to explanation" artifact required by PDPA
#   - Apply: adverse-action notices for declined Singapore loan applicants
#
# PREREQUISITES: 01_shap_global.py (same model, same SHAP bundle).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — how LIME's local linear surrogate works
#   2. Build — LimeTabularExplainer with graceful ImportError handling
#   3. Train — no training; EXPLAIN extreme-risk individual cases
#   4. Visualise — LIME weights and SHAP waterfall side-by-side
#   5. Apply — PDPA adverse-action notices for declined applicants
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv


load_dotenv()



## THEORY — LIME's Local Linear Surrogate


In [ ]:
# LIME (Ribeiro, Singh, Guestrin, 2016) — "Local Interpretable
# Model-agnostic Explanations":
#
# For a single prediction f(x):
#   1. Sample perturbed rows around x (Gaussian for continuous,
#      discrete bin flips for categorical)
#   2. Score each perturbation with the original model f
#   3. Weight perturbations by proximity to x (exponential kernel)
#   4. Fit a SPARSE LINEAR model on the weighted samples
#   5. The linear coefficients ARE the local feature importances
#
# SHAP vs LIME at a glance:
#   SHAP: axioms + exact for trees + consistent across samples
#   LIME: fast + model-agnostic + easier to explain to non-statisticians
#
# Production rule:
#   Tree model   -> TreeSHAP (exact)
#   Black-box    -> LIME or KernelSHAP (approximate)
#   Mixed stack  -> Both; SHAP as ground truth, LIME as sanity check



## TASK 2 — BUILD the LIME explainer


In [ ]:
bundle = build_shap_explainer()
model = bundle["model"]
X_train = bundle["X_train"]
X_test = bundle["X_test"]
y_proba = bundle["y_proba"]
feature_names: list[str] = bundle["feature_names"]
shap_vals: np.ndarray = bundle["shap_vals"]

try:
    from lime.lime_tabular import LimeTabularExplainer

    lime_explainer = LimeTabularExplainer(
        training_data=X_train,
        feature_names=feature_names,
        class_names=["no_default", "default"],
        mode="classification",
        discretize_continuous=True,
        random_state=42,
    )
    HAS_LIME = True
except ImportError:
    lime_explainer = None
    HAS_LIME = False
    print(
        "\n[warn] LIME is not installed in this environment.\n"
        "       Install with: uv pip install lime\n"
        "       The SHAP comparison below still runs.\n"
    )



## TASK 3 — "TRAIN" = pick the most interesting individuals to explain


In [ ]:
risk_order = np.argsort(y_proba)
high_risk_idx = int(risk_order[-1])
low_risk_idx = int(risk_order[0])
borderline_idx = int(risk_order[len(risk_order) // 2])

print_section("Local Explanation Targets")
print(
    f"  Highest risk sample idx: {high_risk_idx}  P(default)={y_proba[high_risk_idx]:.4f}"
)
print(
    f"  Lowest  risk sample idx: {low_risk_idx}  P(default)={y_proba[low_risk_idx]:.4f}"
)
print(
    f"  Borderline      idx:     {borderline_idx}  P(default)={y_proba[borderline_idx]:.4f}"
)



## TASK 4 — VISUALISE local explanations (LIME + SHAP side-by-side)


In [ ]:
if HAS_LIME:
    print_section("LIME Local Weights — HIGHEST-risk applicant", char="─")
    lime_exp = lime_explainer.explain_instance(
        X_test[high_risk_idx],
        model.predict_proba,
        num_features=10,
        top_labels=1,
    )
    for feat_desc, weight in lime_exp.as_list():
        direction = "^risk" if weight > 0 else "v risk"
        print(f"  {feat_desc:<45} {weight:+.4f} ({direction})")

    print_section("SHAP Waterfall — same HIGHEST-risk applicant", char="─")
    sample_shap = shap_vals[high_risk_idx]
    shap_sorted = sorted(
        zip(feature_names, sample_shap, X_test[high_risk_idx]),
        key=lambda t: abs(t[1]),
        reverse=True,
    )[:10]
    for name, sv, fv in shap_sorted:
        direction = "^risk" if sv > 0 else "v risk"
        print(f"  {name:<30} value={fv:>8.2f}  SHAP={sv:+.4f} ({direction})")
else:
    print_section("SHAP Waterfall — HIGHEST-risk applicant", char="─")
    sample_shap = shap_vals[high_risk_idx]
    shap_sorted = sorted(
        zip(feature_names, sample_shap, X_test[high_risk_idx]),
        key=lambda t: abs(t[1]),
        reverse=True,
    )[:10]
    for name, sv, fv in shap_sorted:
        direction = "^risk" if sv > 0 else "v risk"
        print(f"  {name:<30} value={fv:>8.2f}  SHAP={sv:+.4f} ({direction})")


print_section("SHAP Waterfall — LOWEST-risk applicant", char="─")
for name, sv, fv in sorted(
    zip(feature_names, shap_vals[low_risk_idx], X_test[low_risk_idx]),
    key=lambda t: abs(t[1]),
    reverse=True,
)[:8]:
    direction = "^risk" if sv > 0 else "v risk"
    print(f"  {name:<30} value={fv:>8.2f}  SHAP={sv:+.4f} ({direction})")


print_section("SHAP Waterfall — BORDERLINE applicant", char="─")
for name, sv, fv in sorted(
    zip(feature_names, shap_vals[borderline_idx], X_test[borderline_idx]),
    key=lambda t: abs(t[1]),
    reverse=True,
)[:8]:
    direction = "^risk" if sv > 0 else "v risk"
    print(f"  {name:<30} value={fv:>8.2f}  SHAP={sv:+.4f} ({direction})")



### Visual: LIME / SHAP bar chart for highest-risk applicant


In [ ]:
sample_shap_hr = shap_vals[high_risk_idx]
shap_sorted_hr = sorted(
    zip(feature_names, sample_shap_hr),
    key=lambda t: abs(t[1]),
    reverse=True,
)[:10]
fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=[n for n, _ in reversed(shap_sorted_hr)],
        x=[v for _, v in reversed(shap_sorted_hr)],
        orientation="h",
        marker_color=[
            "#ef4444" if v > 0 else "#3b82f6" for _, v in reversed(shap_sorted_hr)
        ],
        name="SHAP value",
    )
)
fig.update_layout(
    title=f"SHAP Waterfall — Highest-Risk Applicant (P(default)={y_proba[high_risk_idx]:.4f})",
    xaxis_title="SHAP value (red = increases risk, blue = decreases risk)",
    yaxis_title="Feature",
    height=max(400, len(shap_sorted_hr) * 35),
)
viz_path = OUTPUT_DIR / "ex6_03_lime_shap_high_risk.html"
fig.write_html(str(viz_path))
print(f"\n  Saved: {viz_path}")



### Checkpoint


In [ ]:
assert (
    y_proba[high_risk_idx] > y_proba[low_risk_idx]
), "Task 4: highest-risk probability must exceed lowest-risk"
# INTERPRETATION: The three waterfalls (high, low, borderline) are the
# concrete artifact a regulator wants to see: each individual decision
# decomposed into per-feature contributions.
print("\n[ok] Checkpoint — local explanations rendered for three risk tiers\n")



## TASK 5 — APPLY: PDPA Adverse-Action Notices for Declined Applicants


In [ ]:
# SCENARIO: Singapore's Personal Data Protection Act (PDPA) and the MAS
# Notice on Credit Decisions require banks to provide a SPECIFIC reason
# — not a generic "did not meet our criteria" — for every declined loan
# application. The explanation must name the top drivers of the decline
# AND must be defensible under audit.
#
# The adverse-action notice template:
#
#     "Dear [applicant], your application for loan #[id] was declined.
#      The top three factors contributing to this decision were:
#
#        1. [feature_1] — your value of [v1] compared to our acceptance
#           range of [range_1]
#        2. [feature_2] — ...
#        3. [feature_3] — ...
#
#      You may contest this decision by contacting our FIDReC liaison."
#
# Why LIME+SHAP together are the right tool:
#   - SHAP provides the LEGALLY DEFENSIBLE decomposition (exact, axiomatic)
#   - LIME provides the CUSTOMER-READABLE linear summary (easier to
#     translate into "your income was too low" narrative)
#   - The two must AGREE on the top-3 drivers before a notice is sent;
#     disagreement flags the application for human review
#
# BUSINESS IMPACT:
#   - ~15,000 declines/month across the bank; PDPA compliance is mandatory
#   - Manual analyst review of declined applications: S$45/application
#     (15,000 * S$45 = S$675,000/month)
#   - SHAP+LIME automated notices reduce analyst review to the 3% of cases
#     where the two methods disagree: 15,000 * 0.03 = 450 manual reviews,
#     costing 450 * S$45 = S$20,250/month
#   - Monthly saving: S$654,750. Annual saving: S$7.85M.
#   - Implementation cost: 4 engineer-weeks (S$32,000). Payback: <1 week.
#
# LIMITATION: LIME's random perturbation sampling makes it unstable —
# running it twice on the same sample can give different top-3 features.
# That's why SHAP is the ground truth and LIME is the plain-English layer.



## REFLECTION


[x] Built a LIME explainer (with graceful fallback if not installed)
  [x] Explained the highest-, lowest-, and borderline-risk applicants
  [x] Compared LIME local weights against SHAP waterfall decomposition
  [x] Designed an adverse-action notice template backed by both methods
  [x] Quantified the PDPA compliance cost savings in S$

  KEY INSIGHT: Global explanations tell regulators how the MODEL works;
  local explanations tell individual customers why THEIR application
  was declined. PDPA requires the latter — one explanation per decision.

  Next: 04_shap_interactions.py — move beyond single-feature effects and
  uncover which FEATURE PAIRS the model uses together.


In [ ]:
print_section("WHAT YOU'VE MASTERED")
print(
)

